In [1]:
from qubic.lib.Qhdf5 import HDF5Dict
import matplotlib.pyplot as plt
import healpy as hp
import numpy as np
from glob import glob
import os
from PIL import Image
import io
from qubic.lib.MapMaking.Qmap_plotter import plot_cross_spectrum
from IPython.display import Image as IPyImage
from qubic.lib.QskySim import get_angular_profile
from copy import deepcopy

In [ ]:
hdf = HDF5Dict()
DATA_PATH = "test_k_max_3_synthbeam_fraction_0.99"

# Check Number of Realisation & Spectra

In [8]:
realisation_path = sorted(glob(os.path.join(DATA_PATH, "Dict", "*.h5")))
spectra_path = realisation_path = sorted(glob(os.path.join(DATA_PATH, "Spectrum", "*.h5")))

print("Number of realisation : ", len(realisation_path))
print("Number of Spectra : ", len(spectra_path))
print("Missing Spectra : ", len(realisation_path) - len(spectra_path))

Number of realisation :  299
Number of Spectra :  299
Missing Spectra :  0


# Check Parameters

In [4]:
def remove_seed(p):
    p = deepcopy(p)  # avoid modifying original dict
    try:
        del p["QUBIC"]["NOISE"]["seed_noise"]
        del p["PLANCK"]["seed_noise"]
    except KeyError:
        pass
    return p

In [5]:
ref_simu_params = remove_seed(hdf.load_dict(realisation_path[0])["parameters"])
ref_spectrum_params = remove_seed(hdf.load_dict(spectra_path[0])["parameters"])

print(ref_simu_params)

{'foldername': 'test_l_min_30', 'filename': 'test', 'CMB': {'cmb': True, 'seed': 1, 'r': 0, 'Alens': 1}, 'Foregrounds': {'Dust': False, 'Synchrotron': False}, 'QUBIC': {'instrument': 'DB', 'configuration': 'FI', 'npointings': 5000, 'nsub_in': 40, 'nsub_out': 16, 'nrec': 2, 'convolution_in': True, 'convolution_out': False, 'bandpass_correction': True, 'interp': False, 'NOISE': {'ndet': 1, 'npho150': 1, 'npho220': 1, 'detector_nep': 4.7e-17, 'duration_150': 3, 'duration_220': 3}, 'SYNTHBEAM': {'synthbeam_kmax': 1, 'synthbeam_fraction': 1}, 'dtheta': 15}, 'SKY': {'nside': 256, 'coverage_cut': 0.2, 'RA_center': 0, 'DEC_center': -57}, 'PLANCK': {'external_data': True, 'weight_planck': 0, 'level_noise_planck': 1, 'bandwidth_planck': 0.2, 'nsub_planck': 100}, 'Pipeline': {'mapmaking': True, 'spectrum': True}, 'PCG': {'n_iter_pcg': 300, 'tol_pcg': 1e-10, 'preconditioner': False, 'initial_guess_intensity_to_zero': True, 'gif': False, 'resolution_plot': 15, 'fwhm_plot': 0}, 'Spectrum': {'dl': 30

In [6]:

for i, real in enumerate(realisation_path):
    d = remove_seed(hdf.load_dict(real)["parameters"])
    test_all_same = (d == ref_simu_params)
    if not test_all_same:
        print(f"Realisation {i} does not have the same parameters than reference one.")
    
for i, spec in enumerate(spectra_path):
    d = remove_seed(hdf.load_dict(spec)["parameters"])
    test_all_same = (d == ref_spectrum_params)
    if not test_all_same:
        print(f"Spectra {i} does not have the same parameters than reference one.")
        
print("If no print above, simulation and spectra parameters are the same across all realisation ! :)")

If no print above, simulation and spectra parameters are the same across all realisation ! :)


In [33]:
stop

NameError: name 'stop' is not defined

# Check X-Spectra

In [ ]:
frames = []
plt.ioff()
for i, file_name in enumerate(sorted(glob(os.path.join(DATA_PATH, "Spectrum", "*.h5")))):
    dict = hdf.load_dict(file_name)
    nus = dict["nus"]
    ell = dict["ell"]
    Dls = dict["Dls"]
    Nls = dict["Nls"]
    Dls_noiseless = Dls - Nls
    N = nus.size - 7

    fig = plot_cross_spectrum(
        nus=nus[:N],
        ell=ell,
        Dl=Dls[:N, :N],
        Dl_err=Nls[:N, :N],
        ymodel=None,
        mode="Dl",
        nrec=N,
        figsize=(12, 10),
        title=f" (QUBIC only) - ID {i}",
    )
    
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf))
    plt.close(fig)

# Save to an in-memory bytes buffer
gif_buffer = io.BytesIO()
frames[0].save(gif_buffer,
                save_all=True,
                append_images=frames[1:],
                duration=100,
                loop=0,
                optimize=True,
                format='GIF')
gif_buffer.seek(0)

# Return an IPython Image object (will be displayed automatically)
IPyImage(data=gif_buffer.getvalue())

# Check Noise profiles

In [ ]:
plt.ioff()
frames = []
Nrec = 2

for i, file_name in enumerate(sorted(glob(os.path.join(DATA_PATH, "Dict", "*.h5")))):
    maps_res = hdf.load_item(file_name, "maps_in_convolved") - hdf.load_item(file_name, "maps")[:Nrec]

    fig = plt.figure(figsize=(12, 8))

    for inu in range(Nrec):
        plt.subplot(1, Nrec, inu + 1)

        get_angular_profile(
            maps_res[inu],
            doplot=True,
            allstokes=True,
            nbins=80,
            thmax=20
        )

        plt.title(f"{nus[inu].round(1)} GHz - ID {i}")

    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf).copy())  # évite bug mémoire
    buf.close()

    plt.close(fig)
    
gif_buffer = io.BytesIO()
frames[0].save(gif_buffer,
                save_all=True,
                append_images=frames[1:],
                duration=50,
                loop=0,
                optimize=True,
                format='GIF')
gif_buffer.seek(0)

# Return an IPython Image object (will be displayed automatically)
IPyImage(data=gif_buffer.getvalue())